# SAE Family Alignment Analysis

This notebook reframes the family-level study as a multi-layer comparison across SAE layers 0, 5, and 11 while reusing the existing cached family profiles.

## Goals
1. Load the three trained SAE checkpoints in a consistent, cache-first way.
2. Compare RNA-type selectivity, enrichment, aligned activation profiles, PCA geometry, and k-NN classification across layers.
3. Keep the plotting cells simple and publication-ready.
4. Leave every figure export line commented so you can save only the figures you want.

## Usage Notes

- Set `DEVICE_PREFERENCE` to `auto`, `cuda`, `mps`, or `cpu`; `auto` selects the best available backend.
- It does not execute automatically here.
- Cached intermediates are stored under `.cache/analysis/family_analysis/...`.
- The notebook requires the standalone 25,000-sequence CD-HIT-2D holdout cache produced by `generate_family_holdout.py`.
- If you want to export a figure, uncomment the corresponding `fig.savefig(...)` line.

In [ ]:
from pathlib import Path
import gc
import importlib
import json
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from scipy.stats import kruskal, rankdata, spearmanr
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    adjusted_rand_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    silhouette_score,
 )
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder

import src.notebook_analysis_utils as notebook_utils

importlib.reload(notebook_utils)

build_type_palette = notebook_utils.build_type_palette
beautify_axes = notebook_utils.beautify_axes
compute_family_feature_statistics = notebook_utils.compute_family_feature_statistics
extract_family_profiles = notebook_utils.extract_family_profiles
load_layer_catalog = notebook_utils.load_layer_catalog
load_layer_model = notebook_utils.load_layer_model
resolve_device = notebook_utils.resolve_device
safe_unload = notebook_utils.safe_unload
set_publication_style = notebook_utils.set_publication_style
sha1_digest = notebook_utils.sha1_digest
summarize_type_distribution = notebook_utils.summarize_type_distribution


def show_and_close(fig, stem: str | None = None) -> None:
    if stem is not None:
        figure_dir = globals().get("FIGURE_DIR")
        if figure_dir is None:
            raise RuntimeError("FIGURE_DIR must be configured before saving figures.")
        fig.savefig(figure_dir / f"{stem}.pdf", format="pdf", bbox_inches="tight")
    plt.show()
    plt.close(fig)
    gc.collect()


def annotate_panel(ax, label: str) -> None:
    """Use the top margin for a compact, consistently aligned layer badge."""
    ax.set_title(
        label,
        loc="right",
        fontsize=12,
        fontweight="bold",
        pad=8,
        color="#374151",
        bbox={
            "boxstyle": "round,pad=0.22",
            "facecolor": "#f3f4f6",
            "edgecolor": "#9ca3af",
            "linewidth": 0.8,
        },
    )


## Configuration

The defaults target the current three trained checkpoints and a reasonably sized disjoint RNAcentral holdout for comparative analysis.

In [ ]:
PROJECT_ROOT = Path.cwd()
ARCHIVE_ROOT = PROJECT_ROOT.parent if (PROJECT_ROOT.parent / "models").is_dir() else PROJECT_ROOT
CHECKPOINT_ROOT = ARCHIVE_ROOT / "models"

LAYER_PATHS = {
    0: CHECKPOINT_ROOT / "layer_00",
    5: CHECKPOINT_ROOT / "layer_05",
    11: CHECKPOINT_ROOT / "layer_11",
}

REFERENCE_LAYER = 0
RUNTIME_TARGET = os.environ.get("SPIRAL_RUNTIME_TARGET", "local")
DEVICE_PREFERENCE = os.environ.get("SPIRAL_DEVICE", "auto")
DEVICE = resolve_device(DEVICE_PREFERENCE)
MODEL_QUANTIZATION = None
USE_CACHE = True
REQUIRE_CACHE = False
CACHE_SCHEMA_VERSION = 4

NUM_SEQUENCES = 25_000
MAX_SEQ_LENGTH = 512
MIN_TYPE_COUNT = 50
HOLDOUT_POLICY_VERSION = "cd_hit_est_2d_80pct_v1"
CD_HIT_EST_2D_IDENTITY = 0.80
CD_HIT_EST_2D_WORD_SIZE = 5
MIN_FEATURE_SEQS = 10
TOP_FEATURES_PER_LAYER = 10
HEATMAP_FEATURE_TRACKS = 10
ACTIVE_THRESHOLD = 0.05
PROJECTION_SAMPLE_SIZE = 6_000
PCA_ACTIVE_FEATURE_CAP = 256
PCA_RANDOM_STATE = 42
PCA_LABEL_LIMIT = 10
KNN_NEIGHBORS = 7

LAYER_COLORS = {
    0: "#0f766e",
    5: "#b45309",
    11: "#7c3aed",
}

ANALYSIS_KEY = sha1_digest(
    {
        "cache_schema_version": CACHE_SCHEMA_VERSION,
        "layer_paths": {key: str(value) for key, value in LAYER_PATHS.items()},
        "model_quantization": MODEL_QUANTIZATION or "checkpoint-default",
        "num_sequences": NUM_SEQUENCES,
        "max_seq_length": MAX_SEQ_LENGTH,
        "min_type_count": MIN_TYPE_COUNT,
        "active_threshold": ACTIVE_THRESHOLD,
        "holdout_policy": HOLDOUT_POLICY_VERSION,
        "cd_hit_est_2d_identity": CD_HIT_EST_2D_IDENTITY,
        "cd_hit_est_2d_word_size": CD_HIT_EST_2D_WORD_SIZE,
    }
)
OUTPUT_ROOT = PROJECT_ROOT / "analysis_outputs" / "family_analysis" / f"multilayer_{ANALYSIS_KEY}"
CACHE_DIR = PROJECT_ROOT / ".cache" / "analysis" / "family_analysis" / f"multilayer_{ANALYSIS_KEY}"
FIGURE_DIR = OUTPUT_ROOT / "figures"
for directory in (OUTPUT_ROOT, CACHE_DIR, FIGURE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

set_publication_style()
warnings.filterwarnings("ignore", category=FutureWarning)

layer_bundles, layer_catalog = load_layer_catalog(LAYER_PATHS)
display(layer_catalog.round(6))
print(f"Runtime target: {RUNTIME_TARGET}")
print(f"Device: {DEVICE} (preference: {DEVICE_PREFERENCE})")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Cache-only mode: {REQUIRE_CACHE}")
print(f"CD-HIT-EST-2D similarity threshold: {CD_HIT_EST_2D_IDENTITY:.0%}")

## Load The RNAcentral Holdout

This subset is built as a disjoint holdout by comparing the training source window and candidate sequences with cd-hit-est-2d at 80% global identity. The first generated result is cached and reused on later runs.

In [ ]:
ARCHIVED_HOLDOUT_ROOT = ARCHIVE_ROOT / "data" / "rnacentral_holdout"
HOLDOUT_EXPORT_ROOT = ARCHIVED_HOLDOUT_ROOT if ARCHIVED_HOLDOUT_ROOT.is_dir() else PROJECT_ROOT / ".cache" / "analysis" / "shared_holdouts"
holdout_stem = f"rnacentral_holdout_{NUM_SEQUENCES}_{MAX_SEQ_LENGTH}_{MIN_TYPE_COUNT}_{HOLDOUT_POLICY_VERSION}"
holdout_audit_stem = f"rnacentral_holdout_audit_{NUM_SEQUENCES}_{MAX_SEQ_LENGTH}_{MIN_TYPE_COUNT}_{HOLDOUT_POLICY_VERSION}"
holdout_paths = {
    "parquet": HOLDOUT_EXPORT_ROOT / f"{holdout_stem}.parquet",
    "audit": HOLDOUT_EXPORT_ROOT / f"{holdout_audit_stem}.json",
}
if not holdout_paths["parquet"].exists() or not holdout_paths["audit"].exists():
    raise FileNotFoundError(
        f"Missing standalone holdout cache: {holdout_paths['parquet']}. "
        "Run `python generate_family_holdout.py` first."
    )
df_data = pd.read_parquet(holdout_paths["parquet"])
holdout_audit = json.loads(holdout_paths["audit"].read_text())
if holdout_audit.get("holdout_policy") != HOLDOUT_POLICY_VERSION:
    raise ValueError(f"Unexpected holdout policy: {holdout_audit.get('holdout_policy')}")
if holdout_audit.get("similarity_filter") != "cd-hit-est-2d":
    raise ValueError("The cached holdout was not generated with CD-HIT-EST-2D.")
if float(holdout_audit.get("similarity_threshold", -1)) != CD_HIT_EST_2D_IDENTITY:
    raise ValueError("The cached holdout was not generated at 80% identity.")
if len(df_data) != NUM_SEQUENCES:
    raise ValueError(f"Expected {NUM_SEQUENCES:,} holdout rows, found {len(df_data):,}.")

if "seq_length" not in df_data.columns:
    df_data["seq_length"] = df_data["sequence"].str.len().astype(np.int32)

type_distribution_df = summarize_type_distribution(df_data)
TYPE_PALETTE = build_type_palette(type_distribution_df["rna_type"].tolist())
display(type_distribution_df.head(20))
print(f"Shared holdout parquet: {holdout_paths['parquet']}")
print(f"Similarity filter: cd-hit-est-2d at {CD_HIT_EST_2D_IDENTITY:.0%} global identity")
if holdout_audit:
    print(
        f"Holdout policy: {holdout_audit.get('holdout_policy', 'unknown')} | "
        f"training rows compared: {holdout_audit.get('training_source_rows', 0):,}"
    )
    print(
        f"Candidate rows scanned: {holdout_audit.get('candidate_rows_scanned', 0):,} | "
        f"returned: {holdout_audit.get('returned_sequences', len(df_data)):,}"
    )
print(f"Retained {len(df_data):,} sequences across {df_data['rna_type'].nunique():,} RNA types")

## Extraction And Cache Helpers

The helper functions below store one profile cache and one results table per layer.


In [ ]:
def family_cache_paths(layer_index: int) -> dict[str, Path]:
    layer_dir = CACHE_DIR / f"layer_{layer_index:02d}"
    layer_dir.mkdir(parents=True, exist_ok=True)
    return {
        "profiles": layer_dir / "family_profiles.npz",
        "results": layer_dir / "feature_family_alignment.csv",
        "summary": layer_dir / "layer_summary.json",
    }


def save_family_layer(layer_index: int, payload: dict, df_results: pd.DataFrame) -> None:
    cache_paths = family_cache_paths(layer_index)
    np.savez_compressed(
        cache_paths["profiles"],
        mean_act_matrix=payload["mean_act_matrix"],
        frac_act_matrix=payload["frac_act_matrix"],
        max_act_matrix=payload["max_act_matrix"],
        bert_mean_matrix=payload["bert_mean_matrix"],
        valid_indices=payload["valid_indices"],
        seq_types=np.array(payload["seq_types"], dtype=object),
    )
    df_results.to_csv(cache_paths["results"], index=False)
    cache_paths["summary"].write_text(
        json.dumps(
            {
                "n_processed": int(payload["n_processed"]),
                "n_skipped": int(payload["n_skipped"]),
            },
            indent=2,
        )
    )


def load_family_layer(
    layer_index: int,
    include_labels: bool = True,
    include_mean_matrix: bool = False,
    include_bert_matrix: bool = False,
    include_auxiliary_matrices: bool = False,
) -> tuple[dict, pd.DataFrame]:
    cache_paths = family_cache_paths(layer_index)
    with np.load(cache_paths["profiles"], allow_pickle=True) as packed:
        payload = {
            "valid_indices": packed["valid_indices"],
        }
        if include_labels:
            payload["seq_types"] = packed["seq_types"].tolist()
        if include_mean_matrix:
            payload["mean_act_matrix"] = packed["mean_act_matrix"]
        if include_bert_matrix:
            payload["bert_mean_matrix"] = packed["bert_mean_matrix"]
        if include_auxiliary_matrices:
            payload["frac_act_matrix"] = packed["frac_act_matrix"]
            payload["max_act_matrix"] = packed["max_act_matrix"]
    df_results = pd.read_csv(cache_paths["results"])
    return payload, df_results


def load_family_projection_layer(layer_index: int) -> dict:
    cache_paths = family_cache_paths(layer_index)
    with np.load(cache_paths["profiles"], allow_pickle=True) as packed:
        return {
            "mean_act_matrix": packed["mean_act_matrix"],
            "bert_mean_matrix": packed["bert_mean_matrix"],
            "seq_types": packed["seq_types"].tolist(),
        }


def summarize_family_layer(layer_index: int, df_results: pd.DataFrame) -> dict:
    if df_results.empty:
        return {
            "layer_index": layer_index,
            "n_features": 0,
            "n_significant": 0,
            "median_eta_squared": np.nan,
            "max_eta_squared": np.nan,
            "median_selectivity": np.nan,
        }
    return {
        "layer_index": layer_index,
        "n_features": int(len(df_results)),
        "n_significant": int(df_results["significant_bonf"].sum()),
        "median_eta_squared": float(df_results["eta_squared"].median()),
        "max_eta_squared": float(df_results["eta_squared"].max()),
        "median_selectivity": float(df_results["selectivity"].median()),
    }


def cross_validated_knn(X: np.ndarray, labels: np.ndarray, n_neighbors: int = KNN_NEIGHBORS) -> dict:
    encoder = LabelEncoder()
    y = encoder.fit_transform(labels)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=PCA_RANDOM_STATE)
    y_true_all = []
    y_pred_all = []

    for train_idx, test_idx in skf.split(X, y):
        knn = KNeighborsClassifier(n_neighbors=n_neighbors, metric="cosine")
        knn.fit(X[train_idx], y[train_idx])
        y_pred = knn.predict(X[test_idx])
        y_true_all.extend(y[test_idx])
        y_pred_all.extend(y_pred)

    y_true = np.array(y_true_all)
    y_pred = np.array(y_pred_all)
    return {
        "classes": encoder.classes_,
        "y_true": y_true,
        "y_pred": y_pred,
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }


def balanced_sample_indices(labels: np.ndarray, sample_size: int, seed: int = PCA_RANDOM_STATE) -> np.ndarray:
    """Draw at most `sample_size // n_types` sequences from each RNA type.

    Types with fewer members than the per-type cap contribute all of them, so the
    result is as close to type-balanced as the holdout permits and is smaller
    than `sample_size`. The unused remainder is deliberately not used to top the
    sample back up to `sample_size`: that pool is dominated by the majority type,
    so drawing from it would reintroduce the imbalance this sampler exists to
    remove.
    """
    rng = np.random.default_rng(seed)
    unique_labels = np.unique(labels)
    if len(unique_labels) == 0:
        return np.array([], dtype=int)

    per_group = max(sample_size // len(unique_labels), 1)
    chosen = []
    for label in unique_labels:
        label_indices = np.where(labels == label)[0]
        take = min(len(label_indices), per_group)
        chosen.extend(rng.choice(label_indices, size=take, replace=False).tolist())

    return np.sort(np.array(sorted(set(chosen)), dtype=int))


def projection_metrics(projection: np.ndarray, labels: np.ndarray, seed: int = PCA_RANDOM_STATE) -> dict[str, float]:
    unique_labels = np.unique(labels)
    if len(unique_labels) < 2 or len(projection) <= len(unique_labels):
        return {"silhouette": np.nan, "ari": np.nan}

    try:
        silhouette = silhouette_score(projection, labels)
    except ValueError:
        silhouette = np.nan

    clusters = KMeans(n_clusters=len(unique_labels), n_init=10, random_state=seed).fit_predict(projection)
    ari = adjusted_rand_score(labels, clusters)
    return {"silhouette": float(silhouette), "ari": float(ari)}

## Compute Or Load Per-Layer Family Profiles

Each layer is processed independently, but every layer reuses the same cached RNAcentral subset and cached family-profile bundle.

In [ ]:
family_feature_tables = {}
family_summary_rows = []

# Sequence profiles are token means, so a feature with bounded activation is
# diluted in proportion to sequence length, and RNA types differ systematically
# in length. RNA-type association is therefore measured after removing each
# feature's rank-dependence on length (see compute_family_feature_statistics).
SEQ_LENGTHS_ALL = df_data["seq_length"].to_numpy().astype(float)

for layer_index, bundle in layer_bundles.items():
    cache_paths = family_cache_paths(layer_index)
    should_use_cache = USE_CACHE and cache_paths["profiles"].exists() and cache_paths["results"].exists()
    if should_use_cache:
        payload_meta, _ = load_family_layer(layer_index, include_labels=False)
        should_use_cache = len(payload_meta["valid_indices"]) > 0
        del payload_meta

    if not should_use_cache and REQUIRE_CACHE:
        raise FileNotFoundError(
            "Cached family alignment artifacts are required but missing for "
            f"layer {layer_index}: {cache_paths['profiles']} and {cache_paths['results']}"
        )

    if should_use_cache:
        payload, _ = load_family_layer(
            layer_index,
            include_labels=True,
            include_mean_matrix=True,
        )
    else:
        embedder, sae, act_mean, act_std = load_layer_model(
            bundle,
            device=DEVICE,
            quantization=MODEL_QUANTIZATION,
        )[1:]
        payload = extract_family_profiles(
            df_data,
            embedder,
            sae,
            act_mean=act_mean,
            act_std=act_std,
            active_threshold=ACTIVE_THRESHOLD,
            max_seq_length=MAX_SEQ_LENGTH,
        )

    df_results = compute_family_feature_statistics(
        payload["mean_act_matrix"],
        payload["seq_types"],
        active_threshold=ACTIVE_THRESHOLD,
        min_feature_seqs=MIN_FEATURE_SEQS,
        nuisance=SEQ_LENGTHS_ALL[payload["valid_indices"]],
    )

    if not should_use_cache:
        save_family_layer(layer_index, payload, df_results)
        safe_unload(embedder, sae)
    else:
        df_results.to_csv(cache_paths["results"], index=False)

    del payload
    gc.collect()

    family_feature_tables[layer_index] = df_results
    family_summary_rows.append(summarize_family_layer(layer_index, df_results))

family_summary_df = pd.DataFrame(family_summary_rows).sort_values("layer_index").reset_index(drop=True)
display(family_summary_df.round(4))

LAYER_ORDER = sorted(layer_bundles)
TYPE_ORDER = [
    rna_type
    for rna_type in type_distribution_df["rna_type"]
    if f"mean_{rna_type}" in family_feature_tables[REFERENCE_LAYER].columns
]
MEAN_COLUMNS = [f"mean_{rna_type}" for rna_type in TYPE_ORDER]
LABELLED_TYPES = TYPE_ORDER[: min(PCA_LABEL_LIMIT, len(TYPE_ORDER))]


def build_family_feature_tracks(
    reference_layer: int = REFERENCE_LAYER,
    top_k: int = HEATMAP_FEATURE_TRACKS,
) -> list[dict]:
    reference_df = family_feature_tables[reference_layer]
    reference_pool = reference_df[reference_df["significant_bonf"]].copy()
    if len(reference_pool) < top_k:
        reference_pool = reference_df.copy()
    reference_pool = reference_pool.sort_values(
        ["eta_squared", "selectivity", "fold_enrichment"],
        ascending=[False, False, False],
    ).head(top_k).reset_index(drop=True)

    reference_profiles = reference_pool[MEAN_COLUMNS].to_numpy(dtype=float)
    reference_norms = np.linalg.norm(reference_profiles, axis=1, keepdims=True) + 1e-12
    normalized_reference = reference_profiles / reference_norms

    tracks = []
    for track_index, (_, row) in enumerate(reference_pool.iterrows()):
        tracks.append(
            {
                "track_index": track_index,
                "reference_feature_idx": int(row["feature_idx"]),
                "reference_type": row["preferred_type"],
                "layers": {reference_layer: row},
            }
        )

    for layer_index, df_layer in family_feature_tables.items():
        if layer_index == reference_layer:
            continue

        candidate_profiles = df_layer[MEAN_COLUMNS].to_numpy(dtype=float)
        candidate_norms = np.linalg.norm(candidate_profiles, axis=1, keepdims=True) + 1e-12
        normalized_candidates = candidate_profiles / candidate_norms
        similarity_matrix = normalized_reference @ normalized_candidates.T

        used_positions = set()
        for track_index, track in enumerate(tracks):
            ranking = np.argsort(similarity_matrix[track_index])[::-1]
            best_position = next(
                (int(position) for position in ranking if int(position) not in used_positions),
                None,
            )
            if best_position is None:
                track["layers"][layer_index] = None
                continue
            used_positions.add(best_position)
            track["layers"][layer_index] = df_layer.iloc[best_position]

    return tracks


family_feature_tracks = build_family_feature_tracks()

## Plot 1: Dataset Overview

These views summarize the type distribution and sequence-length regime of the retained RNAcentral subset.


In [ ]:
top_type_df = type_distribution_df.head(18).copy()
fig, ax = plt.subplots(figsize=(8.8, 6.1))
sns.barplot(
    data=top_type_df,
    y="rna_type",
    x="count",
    palette=[TYPE_PALETTE[rna_type] for rna_type in top_type_df["rna_type"]],
    ax=ax,
)
ax.set_xlabel("Sequence count")
ax.set_ylabel("")
beautify_axes(ax)
fig.tight_layout()
show_and_close(fig, "01_rna_type_distribution")

order = type_distribution_df.head(12)["rna_type"].tolist()
plot_df = df_data[df_data["rna_type"].isin(order)].copy()
fig, ax = plt.subplots(figsize=(8.8, 6.1))
sns.boxenplot(
    data=plot_df,
    y="rna_type",
    x="seq_length",
    palette=[TYPE_PALETTE[rna_type] for rna_type in order],
    ax=ax,
)
ax.set_xlabel("Length (nt)")
ax.set_ylabel("")
beautify_axes(ax)
fig.tight_layout()
show_and_close(fig, "02_sequence_length_by_rna_type")

## Plot 2: Eta-Squared Distribution By Layer

This shows one layer at a time so the distribution shape is easy to compare without squeezing the plots together.

In [ ]:
all_eta_values = np.concatenate([
    family_feature_tables[layer_index]["eta_squared"].to_numpy()
    for layer_index in LAYER_ORDER
])
eta_bins = np.linspace(0.0, max(float(all_eta_values.max()), 1e-3), 50)

for layer_index in LAYER_ORDER:
    eta_values = family_feature_tables[layer_index]["eta_squared"].to_numpy()
    median_value = float(np.median(eta_values))

    fig, ax = plt.subplots(figsize=(8.8, 5.2))
    ax.hist(
        eta_values,
        bins=eta_bins,
        color=LAYER_COLORS[layer_index],
        edgecolor="white",
        alpha=0.88,
        zorder=3,
    )
    ax.axvline(median_value, color="#E63946", linestyle="--", linewidth=2.1, zorder=4)
    ax.text(
        0.98,
        0.96,
        f"Median: {median_value:.3f}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=12,
        fontweight="bold",
        bbox={
            "boxstyle": "round,pad=0.3",
            "facecolor": "white",
            "edgecolor": "#d1d5db",
            "alpha": 0.92,
        },
    )
    ax.set_xlim(0.0, eta_bins[-1])
    ax.set_xlabel("η² (Effect size)")
    ax.set_ylabel("Number of features")
    beautify_axes(ax)
    annotate_panel(ax, f"Layer {layer_index}")
    fig.tight_layout()
    show_and_close(fig, f"03_eta_squared_distribution_layer_{layer_index:02d}")

## Plot 3: Selectivity Distribution By Layer

This shows one layer at a time so the selectivity distribution is readable without cramped multi-panel overlaps.

In [ ]:
chance_level = 1 / len(TYPE_ORDER)
selectivity_bins = np.linspace(chance_level * 0.9, 1.0, 50)

for layer_index in LAYER_ORDER:
    selectivity_values = family_feature_tables[layer_index]["selectivity"].to_numpy()
    median_value = float(np.median(selectivity_values))

    fig, ax = plt.subplots(figsize=(8.8, 5.2))
    ax.hist(
        selectivity_values,
        bins=selectivity_bins,
        color=LAYER_COLORS[layer_index],
        edgecolor="white",
        alpha=0.88,
        zorder=3,
    )
    ax.axvline(chance_level, color="#7c7c7c", linestyle=":", linewidth=1.8, zorder=4)
    ax.axvline(median_value, color="#E63946", linestyle="--", linewidth=2.1, zorder=4)
    ax.text(
        0.98,
        0.96,
        f"Median: {median_value:.3f}\nChance: {chance_level:.3f}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=12,
        fontweight="bold",
        bbox={
            "boxstyle": "round,pad=0.3",
            "facecolor": "white",
            "edgecolor": "#d1d5db",
            "alpha": 0.92,
        },
    )
    ax.set_xlim(chance_level * 0.9, 1.0)
    ax.set_xlabel("Selectivity")
    ax.set_ylabel("Number of features")
    beautify_axes(ax)
    annotate_panel(ax, f"Layer {layer_index}")
    fig.tight_layout()
    show_and_close(fig, f"04_selectivity_distribution_layer_{layer_index:02d}")

## Plot 4: Top 10 Enriched Features Across Layers

This keeps the old enrichment ranking idea, but places all three layers side by side.

In [ ]:
for layer_index in LAYER_ORDER:
    df_layer = family_feature_tables[layer_index]
    enriched_df = df_layer[df_layer["significant_bonf"]].copy()
    if enriched_df.empty:
        enriched_df = df_layer.copy()
    enriched_df = enriched_df.sort_values(
        ["fold_enrichment", "eta_squared"],
        ascending=[False, False],
    ).head(TOP_FEATURES_PER_LAYER)

    y_positions = np.arange(len(enriched_df))
    fig, ax = plt.subplots(figsize=(8.8, 5.2))
    ax.barh(
        y_positions,
        enriched_df["fold_enrichment"],
        color=[TYPE_PALETTE.get(rna_type, "#9ca3af") for rna_type in enriched_df["preferred_type"]],
        edgecolor="white",
        linewidth=0.8,
        zorder=3,
    )
    ax.set_yticks(y_positions)
    ax.set_yticklabels([f"F{feature_idx}" for feature_idx in enriched_df["feature_idx"]])
    ax.invert_yaxis()
    ax.set_xlabel("Fold enrichment")
    ax.set_ylabel("Feature")
    beautify_axes(ax)
    annotate_panel(ax, f"Layer {layer_index}")

    x_offset = enriched_df["fold_enrichment"].max() * 0.02 if not enriched_df.empty else 0.05
    for y_position, (_, row) in zip(y_positions, enriched_df.iterrows()):
        ax.text(
            row["fold_enrichment"] + x_offset,
            y_position,
            f"{row['preferred_type']}  η²={row['eta_squared']:.2f}",
            va="center",
            fontsize=12,
            fontweight="bold",
        )

    fig.tight_layout()
    show_and_close(fig, f"05_top_enriched_features_layer_{layer_index:02d}")

## Plot 5: Top 10 Most Type-Selective SAE Features Across Layers

This keeps the familiar ranked-bar view, but compares the strongest RNA-type selective features across all three layers.

In [ ]:
for layer_index in LAYER_ORDER:
    selective_df = family_feature_tables[layer_index].sort_values(
        ["eta_squared", "selectivity", "fold_enrichment"],
        ascending=[False, False, False],
    ).head(TOP_FEATURES_PER_LAYER)

    y_positions = np.arange(len(selective_df))
    fig, ax = plt.subplots(figsize=(8.8, 5.2))
    ax.barh(
        y_positions,
        selective_df["eta_squared"],
        color=[TYPE_PALETTE.get(rna_type, "#9ca3af") for rna_type in selective_df["preferred_type"]],
        edgecolor="white",
        linewidth=0.8,
        zorder=3,
    )
    ax.set_yticks(y_positions)
    ax.set_yticklabels([f"F{feature_idx}" for feature_idx in selective_df["feature_idx"]])
    ax.invert_yaxis()
    ax.set_xlabel("η² (Effect size)")
    ax.set_ylabel("Feature")
    beautify_axes(ax)
    annotate_panel(ax, f"Layer {layer_index}")

    x_offset = selective_df["eta_squared"].max() * 0.02 if not selective_df.empty else 0.01
    for y_position, (_, row) in zip(y_positions, selective_df.iterrows()):
        ax.text(
            row["eta_squared"] + x_offset,
            y_position,
            f"{row['preferred_type']}  {row['fold_enrichment']:.1f}x",
            va="center",
            fontsize=12,
            fontweight="bold",
        )

    fig.tight_layout()
    show_and_close(fig, f"06_top_selective_features_layer_{layer_index:02d}")

## Plot 6: Aligned Mean Activation Heatmaps Across Layers

Each heatmap column is an aligned feature track anchored in the reference layer so the cross-layer evolution stays easy to read.

In [ ]:
track_summary_rows = []
for track_index, track in enumerate(family_feature_tracks, start=1):
    row = {
        "track": f"T{track_index}",
        "anchor_type": track["reference_type"],
    }
    for layer_index in LAYER_ORDER:
        matched_row = track["layers"].get(layer_index)
        row[f"layer_{layer_index}"] = f"F{int(matched_row['feature_idx'])}" if matched_row is not None else "missing"
    track_summary_rows.append(row)

track_summary_df = pd.DataFrame(track_summary_rows)
display(track_summary_df)

for layer_index in LAYER_ORDER:
    track_rows = [track["layers"].get(layer_index) for track in family_feature_tracks]
    matrix = np.column_stack(
        [
            row[MEAN_COLUMNS].to_numpy(dtype=float) if row is not None else np.zeros(len(MEAN_COLUMNS), dtype=float)
            for row in track_rows
        ]
    )
    column_mean = matrix.mean(axis=0, keepdims=True)
    column_std = matrix.std(axis=0, keepdims=True)
    column_std[column_std == 0] = 1.0
    matrix_z = (matrix - column_mean) / column_std

    fig, ax = plt.subplots(figsize=(max(10.5, len(family_feature_tracks) * 1.15), 5.8))
    sns.heatmap(
        data=matrix_z,
        cmap="RdBu_r",
        center=0.0,
        vmin=-2.5,
        vmax=2.5,
        xticklabels=track_summary_df["track"],
        yticklabels=TYPE_ORDER,
        cbar_kws={"label": "Within-track Z-score"},
        ax=ax,
    )
    ax.set_ylabel("RNA type")
    ax.set_xlabel("Aligned feature track")
    ax.tick_params(axis="x", labelrotation=0, labelsize=12)
    ax.tick_params(axis="y", labelsize=12)
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontweight("bold")
    annotate_panel(ax, f"Layer {layer_index}")
    fig.tight_layout()
    show_and_close(fig, f"07_aligned_mean_activation_heatmap_layer_{layer_index:02d}")

## Plot 7: Effect Of The Length Control (Supplementary)

The RNA-type effect sizes reported throughout are length-controlled. This cell
quantifies what that control removes, by comparing each feature's uncontrolled
eta-squared with the length-controlled value used everywhere else.


In [ ]:
length_control_rows = []

for layer_index in LAYER_ORDER:
    layer_table = family_feature_tables[layer_index]
    retained = layer_table["eta_squared"] / layer_table["eta_squared_uncontrolled"]
    length_control_rows.append(
        {
            "layer_index": layer_index,
            "n_features": len(layer_table),
            "median_eta_uncontrolled": layer_table["eta_squared_uncontrolled"].median(),
            "median_eta_controlled": layer_table["eta_squared"].median(),
            "median_retained": retained.median(),
        }
    )

length_control_df = pd.DataFrame(length_control_rows)
display(length_control_df.round(4))

for layer_index in LAYER_ORDER:
    layer_table = family_feature_tables[layer_index]
    top_uncontrolled = layer_table.nlargest(TOP_FEATURES_PER_LAYER, "eta_squared_uncontrolled")
    y_positions = np.arange(len(top_uncontrolled))
    bar_height = 0.38

    fig, ax = plt.subplots(figsize=(8.8, 5.2))
    ax.barh(
        y_positions - bar_height / 2,
        top_uncontrolled["eta_squared_uncontrolled"],
        height=bar_height,
        color="#0f766e",
        edgecolor="white",
        linewidth=0.8,
        label="Uncontrolled",
        zorder=3,
    )
    ax.barh(
        y_positions + bar_height / 2,
        top_uncontrolled["eta_squared"],
        height=bar_height,
        color="#b45309",
        edgecolor="white",
        linewidth=0.8,
        label="Length-controlled",
        zorder=3,
    )
    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        [f"F{int(row.feature_idx)}  ({row.preferred_type})" for row in top_uncontrolled.itertuples()]
    )
    ax.invert_yaxis()
    ax.set_xlabel("η² (Effect size)")
    ax.set_ylabel("Feature")
    beautify_axes(ax)
    annotate_panel(ax, f"Layer {layer_index}")
    legend = ax.legend(
        title="η²",
        loc="lower center",
        bbox_to_anchor=(0.5, 1.02),
        ncol=2,
        borderaxespad=0.0,
    )
    legend.get_title().set_fontweight("bold")
    for text in legend.get_texts():
        text.set_fontweight("bold")

    fig.tight_layout(rect=(0, 0, 1, 0.90))
    show_and_close(fig, f"11_length_controlled_eta_layer_{layer_index:02d}")


## Plot 7: PCA Of Raw Hidden States Versus SAE Activations

This follows the old family PCA styling, but now compares all three layers and reports silhouette and ARI for each representation.

In [ ]:
projection_rows = []
projection_results = {}

for layer_index in LAYER_ORDER:
    payload = load_family_projection_layer(layer_index)
    labels = np.array(payload["seq_types"])
    sample_indices = balanced_sample_indices(labels, PROJECTION_SAMPLE_SIZE, seed=PCA_RANDOM_STATE)
    label_sample = labels[sample_indices]

    bert_sample = payload["bert_mean_matrix"][sample_indices].astype(np.float32)
    active_feature_mask = (payload["mean_act_matrix"] > ACTIVE_THRESHOLD).any(axis=0)
    if not active_feature_mask.any():
        active_feature_mask = np.ones(payload["mean_act_matrix"].shape[1], dtype=bool)
    sae_sample = payload["mean_act_matrix"][sample_indices][:, active_feature_mask].astype(np.float32)
    if sae_sample.shape[1] > PCA_ACTIVE_FEATURE_CAP:
        variances = sae_sample.var(axis=0)
        top_feature_positions = np.argsort(variances)[-PCA_ACTIVE_FEATURE_CAP:]
        sae_sample = sae_sample[:, top_feature_positions]

    bert_model = PCA(n_components=2, random_state=PCA_RANDOM_STATE)
    sae_model = PCA(n_components=2, random_state=PCA_RANDOM_STATE)
    bert_projection = bert_model.fit_transform(bert_sample)
    sae_projection = sae_model.fit_transform(sae_sample)

    projection_results[layer_index] = {
        "labels": label_sample,
        "BiRNA-BERT": {
            "projection": bert_projection,
            "explained": float(bert_model.explained_variance_ratio_[:2].sum()),
            "metrics": projection_metrics(bert_projection, label_sample, seed=PCA_RANDOM_STATE),
        },
        "SAE": {
            "projection": sae_projection,
            "explained": float(sae_model.explained_variance_ratio_[:2].sum()),
            "metrics": projection_metrics(sae_projection, label_sample, seed=PCA_RANDOM_STATE),
        },
    }

    for representation_name in ["BiRNA-BERT", "SAE"]:
        representation_bundle = projection_results[layer_index][representation_name]
        projection_rows.append(
            {
                "layer_index": layer_index,
                "representation": representation_name,
                "explained_variance": representation_bundle["explained"],
                "silhouette": representation_bundle["metrics"]["silhouette"],
                "ari": representation_bundle["metrics"]["ari"],
            }
        )

    del payload, labels, bert_sample, sae_sample, active_feature_mask
    gc.collect()

projection_metrics_df = pd.DataFrame(projection_rows)
display(projection_metrics_df.round(4))
print("Metric guide: higher is better for explained variance, silhouette, and ARI.")

for layer_index in LAYER_ORDER:
    label_sample = projection_results[layer_index]["labels"]
    for representation_name in ["BiRNA-BERT", "SAE"]:
        fig, ax = plt.subplots(figsize=(7.2, 6.6))
        bundle = projection_results[layer_index][representation_name]
        projection = bundle["projection"]
        metrics = bundle["metrics"]

        for rna_type in TYPE_ORDER:
            mask = label_sample == rna_type
            if not mask.any():
                continue
            ax.scatter(
                projection[mask, 0],
                projection[mask, 1],
                s=26 if rna_type in LABELLED_TYPES else 14,
                alpha=0.9 if rna_type in LABELLED_TYPES else 0.58,
                color=TYPE_PALETTE.get(rna_type, "#9ca3af"),
                edgecolors="#111827" if rna_type in LABELLED_TYPES else "white",
                linewidths=0.32 if rna_type in LABELLED_TYPES else 0.18,
                zorder=3 if rna_type in LABELLED_TYPES else 2,
                rasterized=True,
            )

        legend = fig.legend(
            [plt.Line2D([], [], marker="o", linestyle="", markersize=8,
                        markerfacecolor=TYPE_PALETTE.get(label, "#9ca3af"),
                        markeredgecolor="#111827") for label in LABELLED_TYPES],
            LABELLED_TYPES,
            loc="upper center", bbox_to_anchor=(0.5, 0.985), ncol=5,
            title=f"RNA family · Layer {layer_index} · {representation_name}", frameon=True, columnspacing=0.8,
            handletextpad=0.35, borderaxespad=0.0,
        )
        legend.get_title().set_fontweight("bold")
        for text in legend.get_texts():
            text.set_fontweight("bold")

        metric_text = (
            f"Explained {bundle['explained'] * 100:.1f}%  ·  "
            f"Silhouette {metrics['silhouette']:.3f}  ·  ARI {metrics['ari']:.3f}"
        )
        fig.text(
            0.5,
            0.015,
            metric_text,
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold",
            color="#374151",
        )
        ax.set_xlabel("PC1")
        ax.set_ylabel("PC2")
        ax.tick_params(axis="both", labelsize=12)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontweight("bold")
        ax.grid(alpha=0.18, linewidth=0.4)
        ax.spines[["top", "right"]].set_visible(False)
        fig.tight_layout(rect=(0, 0.075, 1, 0.82))
        representation_slug = representation_name.lower().replace(" ", "_").replace("-", "_")
        show_and_close(fig, f"08_pca_{representation_slug}_layer_{layer_index:02d}")

## PCA–Length Correlations

This diagnostic quantifies how strongly post-preprocessing nucleotide length aligns with each of the two plotted principal components. It uses the exact balanced sample selected for each PCA projection.

In [ ]:
pca_length_rows = []

for layer_index in LAYER_ORDER:
    index_payload, _ = load_family_layer(layer_index, include_labels=False)
    labels = np.asarray(projection_results[layer_index]["labels"])
    full_payload = load_family_projection_layer(layer_index)
    full_labels = np.asarray(full_payload["seq_types"])
    sample_indices = balanced_sample_indices(
        full_labels, PROJECTION_SAMPLE_SIZE, seed=PCA_RANDOM_STATE
    )
    if not np.array_equal(labels, full_labels[sample_indices]):
        raise RuntimeError(f"PCA sample mismatch at layer {layer_index}.")

    valid_indices = np.asarray(index_payload["valid_indices"], dtype=int)
    sample_lengths = SEQ_LENGTHS_ALL[valid_indices][sample_indices]
    for representation_name in ["BiRNA-BERT", "SAE"]:
        projection = projection_results[layer_index][representation_name]["projection"]
        for component_index, component_name in enumerate(["PC1", "PC2"]):
            correlation = float(
                np.corrcoef(projection[:, component_index], sample_lengths)[0, 1]
            )
            pca_length_rows.append(
                {
                    "layer_index": layer_index,
                    "representation": representation_name,
                    "component": component_name,
                    "pearson_r_with_length": correlation,
                    "n_sequences": len(sample_lengths),
                }
            )

    del index_payload, full_payload, full_labels, labels, sample_indices, sample_lengths
    gc.collect()

pca_length_correlation_df = pd.DataFrame(pca_length_rows)
display(pca_length_correlation_df.round(4))

## Plot 8: Cross-Layer k-NN Comparison

This compares macro precision, macro recall, and balanced accuracy for BiRNA-BERT and SAE representations at every layer.

In [ ]:
classification_rows = []
classification_artifacts = {}

for layer_index in LAYER_ORDER:
    payload = load_family_projection_layer(layer_index)
    labels = np.array(payload["seq_types"])
    active_feature_mask = (payload["mean_act_matrix"] > ACTIVE_THRESHOLD).any(axis=0)
    if not active_feature_mask.any():
        active_feature_mask = np.ones(payload["mean_act_matrix"].shape[1], dtype=bool)

    feature_spaces = {
        "BiRNA-BERT": payload["bert_mean_matrix"],
        "SAE": payload["mean_act_matrix"][:, active_feature_mask],
    }
    for representation_name, X in feature_spaces.items():
        result = cross_validated_knn(X, labels)
        classification_artifacts[(layer_index, representation_name)] = result
        classification_rows.append(
            {
                "layer_index": layer_index,
                "layer_label": f"Layer {layer_index}",
                "representation": representation_name,
                "precision_macro": result["precision_macro"],
                "recall_macro": result["recall_macro"],
                "balanced_accuracy": result["balanced_accuracy"],
            }
        )

    del payload, labels, active_feature_mask
    gc.collect()

classification_df = pd.DataFrame(classification_rows).sort_values(["layer_index", "representation"]).reset_index(drop=True)
display(classification_df.round(4))

metric_specs = [
    ("precision_macro", "Macro precision"),
    ("recall_macro", "Macro recall"),
    ("balanced_accuracy", "Balanced accuracy"),
]
palette = {"BiRNA-BERT": "#0f766e", "SAE": "#b45309"}

for figure_index, (column, ylabel) in enumerate(metric_specs, start=1):
    fig, ax = plt.subplots(figsize=(6.4, 5.1))
    sns.barplot(
        data=classification_df,
        x="layer_label",
        y=column,
        hue="representation",
        palette=palette,
        ax=ax,
    )
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    beautify_axes(ax)
    legend = ax.legend(
        title="Representation", loc="lower center",
        bbox_to_anchor=(0.5, 1.02), ncol=2, borderaxespad=0.0,
    )
    legend.get_title().set_fontweight("bold")
    for text in legend.get_texts():
        text.set_fontweight("bold")
    fig.tight_layout(rect=(0, 0, 1, 0.88))
    show_and_close(fig, f"09_knn_{figure_index:02d}_{column}")

## Sequence-Only k-NN Baselines

These controls apply the same five-fold stratified 7-NN protocol to post-preprocessing nucleotide length, mononucleotide frequencies, and their unscaled concatenation. The Euclidean length-only result is also reported because positive one-dimensional vectors are collinear under cosine distance.

In [ ]:
def cross_validated_knn_with_metric(
    X: np.ndarray,
    labels: np.ndarray,
    *,
    metric: str,
    n_neighbors: int = KNN_NEIGHBORS,
) -> dict:
    encoder = LabelEncoder()
    y = encoder.fit_transform(labels)
    splitter = StratifiedKFold(
        n_splits=5, shuffle=True, random_state=PCA_RANDOM_STATE
    )
    y_true_all = []
    y_pred_all = []

    for train_idx, test_idx in splitter.split(X, y):
        knn = KNeighborsClassifier(
            n_neighbors=n_neighbors, metric=metric
        )
        knn.fit(X[train_idx], y[train_idx])
        y_true_all.extend(y[test_idx])
        y_pred_all.extend(knn.predict(X[test_idx]))

    y_true = np.asarray(y_true_all)
    y_pred = np.asarray(y_pred_all)
    return {
        "precision_macro": float(
            precision_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }


reference_payload, _ = load_family_layer(REFERENCE_LAYER, include_labels=True)
reference_indices = np.asarray(reference_payload["valid_indices"], dtype=int)
baseline_labels = np.asarray(reference_payload["seq_types"])
baseline_sequences = (
    df_data.iloc[reference_indices]["sequence"].astype(str).tolist()
)
baseline_lengths = np.asarray(
    [len(sequence) for sequence in baseline_sequences], dtype=np.float64
).reshape(-1, 1)
baseline_composition = np.asarray(
    [
        [sequence.count(base) / len(sequence) for base in "ACGU"]
        for sequence in baseline_sequences
    ],
    dtype=np.float64,
)

baseline_specs = [
    ("Length", baseline_lengths, "cosine"),
    ("Composition", baseline_composition, "cosine"),
    (
        "Length + comp.",
        np.column_stack([baseline_lengths, baseline_composition]),
        "cosine",
    ),
    ("Length", baseline_lengths, "euclidean"),
]
sequence_baseline_rows = []
for representation_name, X, metric in baseline_specs:
    result = cross_validated_knn_with_metric(
        X, baseline_labels, metric=metric
    )
    sequence_baseline_rows.append(
        {
            "representation": representation_name,
            "distance": metric,
            "precision_macro": result["precision_macro"],
            "balanced_accuracy": result["balanced_accuracy"],
        }
    )

sequence_baseline_df = pd.DataFrame(sequence_baseline_rows)
display(sequence_baseline_df.round(4))

## Plot 9: Old-Style Confusion Matrices

These normalized confusion matrices are now shown one layer at a time so the class labels and dominant confusions stay readable for both BiRNA-BERT and SAE.

In [ ]:
representation_order = ["BiRNA-BERT", "SAE"]

for layer_index in LAYER_ORDER:
    n_classes = len(classification_artifacts[(layer_index, representation_order[0])]["classes"])
    panel_size = max(5.8, 0.42 * n_classes)
    fig_size = (panel_size + 1.4, max(6.2, 0.4 * n_classes + 1.8))

    for representation_name in representation_order:
        result = classification_artifacts[(layer_index, representation_name)]
        cm = confusion_matrix(result["y_true"], result["y_pred"], normalize="true")
        label_fontsize = 12
        rotation = 45
        annotation_fontsize = 10
        annotation_threshold = 0.12

        fig, ax = plt.subplots(figsize=fig_size)
        image = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1, interpolation="nearest", aspect="equal")
        tick_positions = np.arange(len(result["classes"]))
        ax.set_xticks(tick_positions)
        ax.set_yticks(tick_positions)
        ax.set_xticklabels(result["classes"], rotation=rotation, ha="right", fontsize=label_fontsize)
        ax.set_yticklabels(result["classes"], fontsize=label_fontsize)
        ax.set_xlabel("Predicted", fontsize=12, fontweight="bold")
        ax.set_ylabel("True", fontsize=12, fontweight="bold")
        ax.tick_params(axis="x", pad=3)
        ax.tick_params(axis="y", pad=3)
        for label in ax.get_xticklabels() + ax.get_yticklabels():
            label.set_fontweight("bold")

        for row_index in range(cm.shape[0]):
            for column_index in range(cm.shape[1]):
                value = cm[row_index, column_index]
                if value < annotation_threshold and not (row_index == column_index and value >= 0.04):
                    continue
                ax.text(
                    column_index,
                    row_index,
                    f"{value:.2f}",
                    ha="center",
                    va="center",
                    fontsize=annotation_fontsize,
                    fontweight="bold",
                    color="white" if value > 0.55 else "black",
                )

        ax.set_title(
            f"Layer {layer_index} · {representation_name}\n"
            f"Balanced accuracy {result['balanced_accuracy']:.3f}  ·  Macro F1 {result['macro_f1']:.3f}",
            fontsize=12,
            fontweight="bold",
            pad=10,
        )
        colorbar = fig.colorbar(image, ax=ax, fraction=0.032, pad=0.02)
        colorbar.set_label("Row-normalized accuracy", fontsize=12, fontweight="bold")
        colorbar.ax.tick_params(labelsize=12)
        for label in colorbar.ax.get_yticklabels():
            label.set_fontweight("bold")
        fig.tight_layout(rect=(0, 0, 1, 0.96))
        representation_slug = representation_name.lower().replace(" ", "_").replace("-", "_")
        show_and_close(fig, f"10_confusion_matrix_layer_{layer_index:02d}_{representation_slug}")

## Cleanup

This final cell is optional, but useful if you want a clean kernel state before any later experiments.

In [ ]:
for name in [
    "projection_results",
    "projection_metrics_df",
    "classification_artifacts",
    "classification_df",
    "track_summary_df",
]:
    globals().pop(name, None)
plt.close("all")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Notebook state is clean.")
